# Pruning

![pruning.png](attachment:pruning.png)

## Introduction

Deep Learning is a highly experimental field, and therefore, many satisfactory (usually non-optimal) solutions may exist for a given problem. Often, neural network architectures are disproportionately large for the complexity of the task. It turns out that we can slim down models with only a minimal loss of prediction accuracy.

**Pruning** a network involves removing individual weights or entire neurons. There are many advantages to this method:
- reducing the network size,
- accelerating inference,
- preventing overfitting,
- improving results.

To effectively reduce the network size, we must zero out a sufficient number of elements in its weight matrices. This allows us to better compress the model in memory. However, simply zeroing out weights is not enough to accelerate inference. We must additionally implement and effectively utilize Sparse Matrix computations. Another pruning method could be removing entire neurons – thereby reducing the actual size of the weight matrices.

In this task, we will focus **only** on **zeroing weights** in the model. **You cannot change the network architecture** (e.g., by removing a neuron or an entire hidden layer). We will consider this problem using **regression** as an example.

## Task

Implement the function `your_pruning_algorithm(model : torch.nn.Module) -> pruned_model: torch.nn.Module`, which takes the model implemented below as input and returns its pruned version - i.e., with the highest possible number of zeroed model parameters (weights and biases), while maintaining the lowest possible Mean Squared Error (MSE) of the prediction.

Below in the notebook, you will find a cell intended for your function. The cells you are supposed to modify will be very clearly marked!

You will be evaluated based on the result of the following function (the higher the value, the better):

$$
\mathrm{score}(s, \epsilon) = \begin{cases}
    0 & \text{if } \epsilon > 1000 \\
    (1 - \frac{\epsilon}{1000})^{1.5} \cdot s ^{1.5} & \text{otherwise}
\end{cases}
$$

where:
- $s$ - the number of zeroed model parameters divided by the number of all model parameters (sparsity)
- $\epsilon$ - the Mean Squared Error on the test set (MSE)

This criterion and all the functions mentioned above are implemented by us below.

## Constraints

- Your function should return the model in a maximum of 5 minutes using Google Colab with GPU.
- The weights file should be saved using the `save_parameters` function under the name `model_parameters.pkl`.
- **You cannot** change the model architecture, i.e., it must have exactly:
    - an input layer of size 128
    - a hidden layer of size 1024
    - a Sigmoid activation function
    - an output layer of size 10

## Submission Files

* This notebook
* Model parameters (weights) saved using the `save_parameters` function. **Do not change** the name of the generated file: `model_parameters.pkl`.

## Evaluation

The weights file you provide will be evaluated. However, you should also provide a working notebook that, after executing all cells with the `FINAL_EVALUATION_MODE` flag set to `True`, will produce the `model_parameters.pkl` weights file in under 5 minutes (measured on Google Colab with GPU access).

You can score between 0 and 1.5 points for this task. If you get a `score` below 0.085, you will receive 0 points, and if above 0.95, you will receive 1.5 points. Between these values, your result increases linearly with the `score` value.

# Starter Code

In [ ]:
FINAL_EVALUATION_MODE = False  # During the evaluation, we will set this flag to True.

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
import copy
import pickle

import numpy as np
from IPython.display import clear_output
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import SGD

np.random.seed(0)
torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using {device} device")

Using cuda device


## Data Loading

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

# Function to load training and validation data as np.array
def load_data_from_file(x_train_path, y_train_path, x_valid_path, y_valid_path):
    X_train = np.load(x_train_path)
    y_train = np.load(y_train_path)

    X_valid = np.load(x_valid_path)
    y_valid = np.load(y_valid_path)

    return X_train, y_train, X_valid, y_valid


# Dataset class
class InMemDataset(Dataset):
    def __init__(self, xs, ys, device='cpu'):
        super().__init__()
        self.dataset = []
        for i in tqdm(range(len(xs))):
            self.dataset.append((torch.tensor(xs[i]).to(device).float(), torch.tensor(ys[i]).to(device).float() ))

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

# Let's load the data and create dataloaders
X_train, y_train, X_valid, y_valid = load_data_from_file(
    "train_data/X_train.npy",
    "train_data/y_train.npy",
    "valid_data/X_valid.npy",
    "valid_data/y_valid.npy",
)

batch_size = 128

_train = InMemDataset(X_train, y_train, device)

_valid = InMemDataset(X_valid, y_valid, device)

loaders = {
    "train" : DataLoader(_train, batch_size=batch_size, shuffle=True),
    "valid" : DataLoader(_valid, batch_size=batch_size, shuffle=False),
}

  0%|          | 0/8000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

## Code with evaluation criteria

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

# Total criterion defined in the task description
def score(mse_loss, sparsity, mse_weight=1.5, sparsity_weight=1.5):

    if type(mse_loss) == np.ndarray:
        mse_loss[mse_loss > 1000] = 1000
    else:
        if mse_loss > 1000:
            mse_loss = 1000

    score = (1 - mse_loss / 1000) ** mse_weight * sparsity**sparsity_weight
    return score

# Ratio of zeroed weights to all weights
def get_sparsity(model):
    total_params = 0
    zero_params = 0

    for name, param in model.named_parameters():
        if "weight" in name or "bias" in name:
            total_params += param.numel()
            zero_params += torch.sum(param == 0).item()

    sparsity = zero_params / total_params
    return sparsity


# Mean Squared Error (MSE)
def compute_error(model, data_loader):
    model.eval()

    losses = 0
    num_of_el = 0
    with torch.no_grad():
        for x, y in data_loader:
            outputs = model(x)
            num_of_el += x.shape[0] * y.shape[1]
            losses += model.loss(outputs, y, reduction="sum")

    return losses / num_of_el


def points(score):
    def scale(x, lower=0.085, upper=0.95, max_points=1.5):
        scaled = min(max(x, lower), upper)
        return (scaled - lower) / (upper - lower) * max_points
    return scale(score)

## Model Architecture

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

# Let's define the architecture of our network
class MLP(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(128, 1024),
            nn.Sigmoid(),
            nn.Linear(1024, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.layers(x)
        return logits

    def loss(self, input, target, reduction="mean"):
        mse_loss = nn.MSELoss(reduction=reduction)
        return mse_loss(input, target)

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

# Network weights initialization
def init_weights(m):
    ''' Initialize the weights in the module m.'''
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_normal_(m.weight)
        m.bias.data.fill_(0.01)


# Function to save model weights to a file - remember that your weights file must be named: model_parameters.pkl
def save_parameters(model, file_name="model_parameters.pkl", to_file=True):

    params_to_save = {}
    for name, param in model.named_parameters():
        params_to_save[name] = param.to("cpu")

    if not to_file:
        return params_to_save

    with open(f"{file_name}", "wb") as f:
        pickle.dump(params_to_save, f)


# Function to load model weights from a file
def load_parameters(model, file_name="model_parameters.pkl", from_file=True, params=None):

    if from_file:
        with open(f"{file_name}", "rb") as f:
            params_to_load = pickle.load(f)
    else:
        params_to_load = params

    for name, param in model.named_parameters():
        with torch.no_grad():
            param[...] = params_to_load[name].to(device)

## Model Training

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

# Function to train the model
def train_model(model: nn.Module,
              data_loaders: dict[str, DataLoader],
              num_epochs: int,
              optimizer: torch.optim.Optimizer,
              verbose: bool =True
              ) -> tuple[torch.Tensor, float]:

    """Function to train the model.

    Args:
        model (torch.nn.Module): Neural network to train.
        data_loaders (dict[str, DataLoader]): Dictionary containing DataLoaders for training and validation sets.
        num_epochs (int): Number of epochs for training.
        optimizer (torch.optim.Optimizer): Optimizer for training the model.
        verbose (bool, optional): If True, shows training progress.

    Returns:
        tuple[torch.Tensor, float]: Tuple containing the best set of model parameters found during training and the corresponding loss value on the validation set.
    """
    if FINAL_EVALUATION_MODE:
        verbose = False

    best_epoch = None
    best_params = None
    best_val_loss = np.inf

    for epoch in range(num_epochs):
        model.train()
        _iter = 1
        for inputs, targets in data_loaders['train']:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = model.loss(outputs, targets)
            loss.backward()
            optimizer.step()

            if verbose:
                if _iter % 10 == 0:
                    print(f"Minibatch {_iter:>6}    |  loss {loss.item():>5.2f}  |")

            _iter +=1

        val_loss = compute_error(model, data_loaders["valid"])

        if val_loss < best_val_loss:
            best_epoch = epoch
            best_val_loss = val_loss
            best_params = [copy.deepcopy(p.detach().cpu()) for p in model.parameters()]

        if verbose:
            clear_output(True)
            m = f"After epoch {epoch:>2} | valid loss: {val_loss:>5.2f}"
            print("{0}\n{1}\n{0}".format("-" * len(m), m))

    if best_params is not None:
        if verbose:
            print(f"\nLoading best params on validation set in epoch {best_epoch} with loss {best_val_loss:.2f}")
        with torch.no_grad():
            for param, best_param in zip(model.parameters(), best_params):
                param[...] = best_param

    return best_params, best_val_loss

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

initial_model = MLP().to(device)
initial_model.apply(init_weights)

optimizer = SGD(
    initial_model.parameters(),
    lr = 0.01,
    momentum = 0.95,
    weight_decay = 0.001)

best_params, best_val_loss = train_model(initial_model, loaders, num_epochs=100, optimizer=optimizer, verbose=True)

loss = compute_error(initial_model, loaders["valid"])
m = f"| Validation loss: {loss:>5.2f} |"
print("{0}\n{1}\n{0}".format("-" * len(m), m))

-----------------------------------
After epoch 99 | valid loss: 506.40
-----------------------------------

Loading best params on validation set in epoch 93 with loss 494.30
---------------------------
| Validation loss: 494.30 |
---------------------------


## Example Solution
Below we present a simple solution which is obviously not optimal. 
It serves only to demonstrate how the entire notebook should function.

In [ ]:
def starter_pruning_algorithm(model):
    with torch.no_grad():
        model.layers[0].weight[:, 0:2] = 0
    return model

In [ ]:
if not FINAL_EVALUATION_MODE:
    # Let's make a deep copy so as not to change the weights of the trained model
    model_to_prune = copy.deepcopy(initial_model)

    # Let's prune the weights with the example solution
    model_to_prune = starter_pruning_algorithm(model_to_prune)

    # Saving model parameters (here we changed the file name, you must save it as "model_parameters.pkl")
    save_parameters(model_to_prune, "starter_model_parameters.pkl")

    # Let's see how to load previously saved weights into a newly created model
    new_model = MLP().to(device)
    loss = compute_error(new_model, loaders["valid"])
    print(f"The new model has a loss of {loss:.3f}")

    # Loading model parameters
    load_parameters(new_model, "starter_model_parameters.pkl")
    loss = compute_error(new_model, loaders["valid"])
    print(f"After loading the parameters, the model has a loss of {loss:.3f}")

    mse = compute_error(new_model, loaders["valid"])
    sparsity = get_sparsity(new_model)

    print(f"Model MSE: {mse:.3f} Sparsity: {sparsity:.3f}")
    model_score = score(mse, sparsity)
    print(f"Your model's score is {model_score:.3f}!")
    print(f"Your solution scores {points(model_score):.3f}/1.5 points!")

Nowy model ma loss 31210.521
Po wczytaniu parametrów model ma loss 498.001
MSE modelu: 498.001 Sparsity: 0.014
Wynik twojego modelu to 0.001!
Twoje rozwiązanie zdobywa 0.000/1.5 punktów!


# Your Solution
This section is the only place where you can change the code!

In [ ]:
def your_pruning_algorithm(model):
    import torch.nn.utils.prune as prune
    from torch.optim import Adam
    model.to(device)

    # Threshold for initial magnitude pruning
    C = 2.3

    loader = DataLoader(_train, batch_size=128, shuffle=True)

    with torch.no_grad():
        mask = model.layers[0].weight
        mask = torch.abs(mask)
        mask = mask > C
        model.layers[0].weight *= mask

        mask = model.layers[0].bias
        mask = torch.abs(mask)
        mask = mask > C
        model.layers[0].bias *= mask

        mask = model.layers[2].weight
        mask = torch.abs(mask)
        mask = mask > C
        model.layers[2].weight *= mask

        mask = model.layers[2].bias
        mask = torch.abs(mask)
        mask = mask > C
        model.layers[2].bias *= mask

    # Fine-tuning threshold
    C = 0.05
    epochs = 3000

    for epoch in range(epochs):
        model.train()
        optimizer = Adam(model.parameters(), lr=0.005)
        for index, data in enumerate(loader):
            optimizer.zero_grad()
            inputs, targets = data
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)
            loss = model.loss(outputs, targets)
            loss.backward()

            optimizer.step()

            # Enforce sparsity after gradient update
            with torch.no_grad():
                mask = model.layers[0].weight
                mask = torch.abs(mask)
                mask = mask > C
                model.layers[0].weight *= mask

                mask = model.layers[0].bias
                mask = torch.abs(mask)
                mask = mask > C
                model.layers[0].bias *= mask

                mask = model.layers[2].weight
                mask = torch.abs(mask)
                mask = mask > C
                model.layers[2].weight *= mask

                mask = model.layers[2].bias
                mask = torch.abs(mask)
                mask = mask > C
                model.layers[2].bias *= mask

    # Saving model parameters
    save_parameters(model, "model_parameters.pkl")
    return model

model_to_prune = copy.deepcopy(initial_model)
your_pruning_algorithm(model_to_prune)

MLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layers): Sequential(
    (0): Linear(in_features=128, out_features=1024, bias=True)
    (1): Sigmoid()
    (2): Linear(in_features=1024, out_features=10, bias=True)
  )
)

## Evaluation

The following code will be used to evaluate the solution. After submitting the solution to us, the evaluate_algorithm(X_valid, y_valid) function will be executed, i.e., almost identical code as below will run on the test_data directory available only to the graders.

Ensure before submitting that the entire notebook (also with the FINAL_EVALUATION_MODE = True flag set) executes from start to finish without errors, without user intervention, and saves the weights in the model_parameters.pkl file after executing the Run All command. Also, check if validation_script.py returns the expected result.

In [ ]:
def evaluate(X_test, y_test):
    """Validator"""
    test_model = MLP().to(device)
    load_parameters(test_model)

    batch_size = 128

    _test = InMemDataset(X_test, y_test, device)
    test_loader = DataLoader(_test, batch_size=batch_size, shuffle=False)

    mse = compute_error(test_model, test_loader)
    sparsity = get_sparsity(test_model)

    print(f"Model had error: {mse:.3f} and sparsity: {sparsity:.3f}")
    model_score = score(mse, sparsity)

    return model_score

In [ ]:
if not FINAL_EVALUATION_MODE:
    model_score = evaluate(X_valid, y_valid)
    print(f"Your solution gets score {model_score:.3f} on validation set.")
    print(f"Your solution gets {points(model_score):.3f}/1.5 points on validation set!")

  0%|          | 0/2000 [00:00<?, ?it/s]

Model had error: 9.303 and sparsity: 0.957
Your solution gets score 0.923 on validation set.
Your solution gets 1.454/1.5 points on validation set!
